In [ ]:
%stop_session

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

In [ ]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE         = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO       = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE         = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO       = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD                    = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV                = f"{BASE}/gold/gmv_daily_by_subsidiary"

## BRONZE — product_item (eventos CDC, colunas das prints)

In [ ]:
# Criar DataFrame diretamente (dados do enunciado - Exercício 2)
df_product_item_bronze = spark.createDataFrame([
    ("2023-01-20 22:02:00", "2023-01-20", 55, 696969, 10, 50.00),
    ("2023-01-25 23:59:59", "2023-01-25", 56, 808080, 120, 2400.00),
    ("2023-02-26 03:00:00", "2023-02-26", 69, 373737, 2, 2000.00),
    ("2023-07-12 09:00:00", "2023-07-12", 55, 696969, 10, 55.00)
], schema=[
    "transaction_datetime",
    "transaction_date",
    "purchase_id",
    "product_id",
    "item_quantity",
    "purchase_value"
])

df_product_item_bronze = df_product_item_bronze \
    .withColumn("transaction_datetime", col("transaction_datetime").cast("timestamp")) \
    .withColumn("transaction_date", col("transaction_date").cast("date")) \
    .withColumn("purchase_id", col("purchase_id").cast("bigint")) \
    .withColumn("product_id", col("product_id").cast("bigint")) \
    .withColumn("item_quantity", col("item_quantity").cast("int")) \
    .withColumn("purchase_value", col("purchase_value").cast(DecimalType(18, 2)))

df_product_item_bronze = df_product_item_bronze.withColumn(
    "ingestion_date",
    to_utc_timestamp(current_timestamp(), "UTC")
)


# Persistir Bronze em DELTA (append-only)
df_product_item_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("transaction_date") \
    .save(PATH_BRONZE_PRODUCT_ITEM)

In [ ]:
# uat
df_product_item_bronze.createOrReplaceTempView("vw_product_item_bronze")

spark.sql("SELECT * FROM vw_product_item_bronze").show()